In [74]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq

groq_api_key = os.getenv("GROQ_API_KEY")
model = ChatGroq(model = "openai/gpt-oss-20b", groq_api_key = groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002398720D160>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000239872AF860>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [75]:
from langchain_core.messages import HumanMessage 

model.invoke([
    HumanMessage(content="Hello my name is shaurya , I m cheif AI Engineer")
])

AIMessage(content='Hello Shaurya! 👋 It’s great to meet the Chief AI Engineer. How can I assist you today? Whether you’re looking for insights, brainstorming ideas, or tackling a specific project, I’m here to help.', additional_kwargs={'reasoning_content': 'User says "Hello my name is shaurya , I m cheif AI Engineer". Probably greeting. We respond politely. Also note that the user says "cheif AI Engineer" maybe "chief". We can correct or just respond. Also we should ask if they need help.'}, response_metadata={'token_usage': {'completion_tokens': 113, 'prompt_tokens': 85, 'total_tokens': 198, 'completion_time': 0.124617352, 'completion_tokens_details': {'reasoning_tokens': 58}, 'prompt_time': 0.004117433, 'prompt_tokens_details': None, 'queue_time': 0.210826354, 'total_time': 0.128734785}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a12402de73', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07ae8-50e7-

In [76]:
from langchain_core.messages import AIMessage
model.invoke([
    HumanMessage(content="Hello my name is shaurya , I m cheif AI Engineer"),
    AIMessage(content="Hello shaurya, nice to meet you! How can I assist you today?"),
    HumanMessage(content="Hey Whats My name and what do I do")
])


AIMessage(content='You’re **Shaurya**, and you’re the **Chief AI Engineer**. That means you’re the lead on AI strategy, design, and implementation—guiding teams, shaping AI products, and ensuring cutting‑edge solutions get built and deployed. 🚀', additional_kwargs={'reasoning_content': 'User says "Hey Whats My name and what do I do". The user earlier said "Hello my name is shaurya , I m cheif AI Engineer". So name is shaurya. They are chief AI Engineer. We should respond accordingly. Also maintain the style guidelines. Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 123, 'prompt_tokens': 121, 'total_tokens': 244, 'completion_time': 0.194498783, 'completion_tokens_details': {'reasoning_tokens': 61}, 'prompt_time': 0.005976642, 'prompt_tokens_details': None, 'queue_time': 0.211839367, 'total_time': 0.200475425}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a8c584dda7', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'mod

# Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of
inputs and outputs of the model, and store them in some datastore. Future interactions will then
load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [77]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory


store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store :
        store[session_id] = ChatMessageHistory()
    
    return store[session_id]
        
with_message_history = RunnableWithMessageHistory(model,get_session_history)


c:\Agentic_AI_Learning\main_stuff\.conda\py312\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [78]:
config = {"configurable": {"session_id": "chat1"}}

In [79]:
response = with_message_history.invoke([
        HumanMessage(content="Hello my name is shaurya , I m cheif AI Engineer")],
        config=config
)

In [80]:
response.content 

'Hello Shaurya! Great to meet the Chief AI Engineer. How can I assist you today?'

In [81]:
res = with_message_history.invoke([
    HumanMessage(content="what is my name and what do i do ?")
], config=config)

In [82]:
res.content

'You’re Shaurya, and you work as a **Chief AI Engineer**.'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the
raw user input is just a message, which we are passing to the LLM. Letis now make that a bit more complicated.
First, let's add in a system message with some custom instructions (but still taking messages as input). Next,
we'll add in more input besides just the messages.

In [83]:
from langchain_core.prompts import ChatPromptTemplate , MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "you are a helpful assistant , answer all the questions in the best of your abilities "),
    MessagesPlaceholder(variable_name="messages"),
])

chain = prompt | model

In [84]:
chain.invoke(
    {"messages": [HumanMessage(content="Hello my name is shaurya , I m cheif AI Engineer")]}
)

AIMessage(content='Hello Shaurya! 👋 It’s great to meet the Chief AI Engineer. How can I assist you today? Whether it’s brainstorming AI concepts, reviewing code, discussing best practices, or anything else, just let me know!', additional_kwargs={'reasoning_content': 'We have a user greeting: "Hello my name is shaurya , I m cheif AI Engineer". They haven\'t asked a question. We should respond politely, perhaps ask how we can help. No conflict.'}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 105, 'total_tokens': 205, 'completion_time': 0.109794864, 'completion_tokens_details': {'reasoning_tokens': 44}, 'prompt_time': 0.052414727, 'prompt_tokens_details': None, 'queue_time': 0.322113254, 'total_time': 0.162209591}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d3e146e1a5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07ae8-58af-7373-830d-8ccb41213542-0', tool_calls=[], inv

In [85]:
with_message_history = RunnableWithMessageHistory(chain,get_session_history)

c:\Agentic_AI_Learning\main_stuff\.conda\py312\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [86]:
config = {"configurable": {"session_id": "chat2"}}
response = with_message_history.invoke([
    HumanMessage(content="Hello my name is shaurya , I m cheif AI Engineer")],
    config=config
)
response 

AIMessage(content='Hello Shaurya! 👋 It’s great to meet the Chief AI Engineer. How can I assist you today? Whether it’s brainstorming new AI strategies, troubleshooting models, or exploring cutting‑edge research, I’m here to help. Just let me know what you’re working on or what questions you have!', additional_kwargs={'reasoning_content': 'We need to respond politely, maybe ask how we can help, mention expertise.'}, response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 105, 'total_tokens': 194, 'completion_time': 0.116482458, 'completion_tokens_details': {'reasoning_tokens': 17}, 'prompt_time': 0.032413419, 'prompt_tokens_details': None, 'queue_time': 0.311148536, 'total_time': 0.148895877}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_4f7e7dc26e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a07ae8-5b1a-7251-998f-061796c49c4d-0', tool_calls=[], invalid_tool_calls=[], usage_metada

In [87]:
response.content

'Hello Shaurya! 👋 It’s great to meet the Chief AI Engineer. How can I assist you today? Whether it’s brainstorming new AI strategies, troubleshooting models, or exploring cutting‑edge research, I’m here to help. Just let me know what you’re working on or what questions you have!'

In [88]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

config = {"configurable": {"session_id": "chat2"}}

c:\Agentic_AI_Learning\main_stuff\.conda\py312\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [89]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="Hello my name is shaurya , I m cheif AI Engineer")]},
    config=config
)
response.content

'Hi Shaurya! 👋 It’s a pleasure to meet the Chief AI Engineer. How can I support your work today? Whether you’re tackling a new project, debugging a model, or just curious about the latest AI trends, feel free to let me know!'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left
unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the
LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

In [90]:
from langchain_core.messages import SystemMessage , trim_messages

trimmer = trim_messages(
    max_tokens=45 ,
    strategy="last"  , 
    token_counter="approximate" , 
    include_system=True , 
    allow_partial= False , 
    start_on = "human")

messages = [
    SystemMessage(content="you are a helpful assistant , answer all the questions in the best of your abilities "),
    HumanMessage(content="Hello my name is shaurya , I m cheif AI Engineer"),
    AIMessage(content="Hello shaurya, nice to meet you! How can I assist you today?"),
    HumanMessage(content="Hey Whats My name and what do I do")
]

trimmer.invoke(messages)



[SystemMessage(content='you are a helpful assistant , answer all the questions in the best of your abilities ', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hey Whats My name and what do I do', additional_kwargs={}, response_metadata={})]

In [91]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    |prompt 
    |model
)

chain.invoke(
    {"messages": messages + [HumanMessage(content="Hey what do I do and what is my name")]}
)

AIMessage(content='Hey there! I’m ChatGPT, your friendly AI helper. I don’t have any personal info about you, so I can’t tell you your name—if you’d like, just let me know what you’re called and I’ll remember it for our chat.\n\nAs for “what do I do,” that depends on the situation. If you’re looking for guidance on a task, a project, or just wondering what to tackle next, feel free to give me a bit more context—what’s the goal or problem you’re facing? Once I know that, I can suggest steps, resources, or ideas to help you move forward. Let me know what you’d like to work on!', additional_kwargs={'reasoning_content': 'User: "Hey what do I do and what is my name". They ask what they do? It\'s ambiguous. They might be asking what to do in some context. They also ask what is my name. They likely want the assistant to respond with a friendly answer, maybe guess their name? But we don\'t have any info. We can respond: "I\'m ChatGPT. I don\'t know your name, but you can tell me." Also ask wha

In [92]:
## Lets wrap it with message history

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

config = {"configurable": {"session_id": "chat3"}}
res = with_message_history.invoke(
    {"messages": messages + [HumanMessage(content="Hey what do I do and what is my name")]},
    config=config
)

res.content

c:\Agentic_AI_Learning\main_stuff\.conda\py312\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


'I’d love to help, but I’m not sure what you’re referring to. Could you tell me a bit more about what you’re trying to accomplish or the situation you’re in? That way I can give you the best advice. And if you’d like me to call you by a particular name, just let me know!'